In [ ]:
%%capture
!git clone --single-branch --branch fast_tokenizers_BARTpho_PhoBERT_BERTweet https://github.com/datquocnguyen/transformers.git

In [ ]:
cd transformers

In [ ]:
%%capture
!pip3 install -e .

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

In [ ]:
tokenizer.is_fast

In [ ]:
import os
import ast
from datasets import load_dataset, Dataset
import pandas as pd
from huggingface_hub import login

login("your_huggingface_auth_token_here")

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# We used this dataset from BOSCH@AI_Team (https://huggingface.co/datasets/QuangTran276/new_reasoning), but for convenient, we had re-preprocessed this dataset into a standard format.

df = pd.read_csv("./datasets/processed/new_reasoning/mcq_reasoning.csv")
df_val = df.sample(146, random_state=42)
df_train = df.drop(index=df_val.index)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = pd.read_parquet("hf://datasets/VLSP2025-LegalSML/Public-Test/multichoice_questions/train-00000-of-00001.parquet")
len(df_train), len(df_val), len(df_test)

In [ ]:
df_train.dropna(inplace=True)
df_train = df_train.reset_index(drop=True)
df_train

In [ ]:
def formatting_choice(choices):
    choices = list(choices)
    option_a = choices[0]
    option_b = choices[1]
    option_c = choices[2]
    option_d = choices[3]

    list_options = {"A": option_a, "B": option_b, "C": option_c, "D": option_d}
    return list_options

df_test["choices_std"] = df_test["choices"].apply(formatting_choice)
df_test

In [ ]:
def num_to_choice(n: int) -> str:
    mapping = ["A", "B", "C", "D"]
    return mapping[n]

df_test["answer_std"] = df_test["answer"].apply(num_to_choice)
df_test

In [ ]:
df_test = df_test[["question", "choices_std", "answer_std"]]
df_test.columns = ["question", "choices", "answer"]
df_test

In [ ]:
df_train["answer"].value_counts()

In [ ]:
df_val["answer"].value_counts()

In [ ]:
df_test["answer"].value_counts()

In [ ]:
df_train["option_A"] = df_train["choices"].apply(lambda x: ast.literal_eval(x).get("A"))
df_train["option_B"] = df_train["choices"].apply(lambda x: ast.literal_eval(x).get("B"))
df_train["option_C"] = df_train["choices"].apply(lambda x: ast.literal_eval(x).get("C"))
df_train["option_D"] = df_train["choices"].apply(lambda x: ast.literal_eval(x).get("D"))

df_val["option_A"] = df_val["choices"].apply(lambda x: ast.literal_eval(x).get("A"))
df_val["option_B"] = df_val["choices"].apply(lambda x: ast.literal_eval(x).get("B"))
df_val["option_C"] = df_val["choices"].apply(lambda x: ast.literal_eval(x).get("C"))
df_val["option_D"] = df_val["choices"].apply(lambda x: ast.literal_eval(x).get("D"))

df_test["choices"] = df_test["choices"].astype(str)

df_test["option_A"] = df_test["choices"].apply(lambda x: ast.literal_eval(x).get("A"))
df_test["option_B"] = df_test["choices"].apply(lambda x: ast.literal_eval(x).get("B"))
df_test["option_C"] = df_test["choices"].apply(lambda x: ast.literal_eval(x).get("C"))
df_test["option_D"] = df_test["choices"].apply(lambda x: ast.literal_eval(x).get("D"))

df_train = df_train[["question", "option_A", "option_B", "option_C", "option_D", "answer"]]
df_val = df_val[["question", "option_A", "option_B", "option_C", "option_D", "answer"]]
df_test = df_test[["question", "option_A", "option_B", "option_C", "option_D", "answer"]]

In [ ]:
df_train.columns = ["sent1", "ending0", "ending1", "ending2", "ending3", "label"]
df_val.columns = ["sent1", "ending0", "ending1", "ending2", "ending3", "label"]
df_test.columns = ["sent1", "ending0", "ending1", "ending2", "ending3", "label"]

mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
df_train['label'] = df_train['label'].map(mapping)
df_val['label'] = df_val['label'].map(mapping)
df_test['label'] = df_test['label'].map(mapping)

In [ ]:
!pip install pyvi -q

In [ ]:
dataset_train = Dataset.from_pandas(df_train)
dataset_val = Dataset.from_pandas(df_val)
dataset_test = Dataset.from_pandas(df_test)

In [ ]:
df_test

In [ ]:
dataset_train[0]

In [ ]:
from pyvi import ViTokenizer

def tokenize_function(example):
    example["sent1"] = ViTokenizer.tokenize(example["sent1"])
    example["ending0"] = ViTokenizer.tokenize(example["ending0"])
    example["ending1"] = ViTokenizer.tokenize(example["ending1"])
    example["ending2"] = ViTokenizer.tokenize(example["ending2"])
    example["ending3"] = ViTokenizer.tokenize(example["ending3"])
    return example

dataset_train = dataset_train.map(tokenize_function)
dataset_val = dataset_val.map(tokenize_function)
dataset_test = dataset_test.map(tokenize_function)

In [ ]:
ending_names = ["ending0", "ending1", "ending2", "ending3"]

def preprocess_function(examples):
    question_headers = examples["sent1"]
    second_sentences = [
        [f"{header} {examples[end][i]}" for end in ending_names] 
        for i, header in enumerate(question_headers)
    ]

    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        second_sentences,
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    return {
        k: [v[i : i + 4] for i in range(0, len(v), 4)]
        for k, v in tokenized_examples.items()
    }

In [ ]:
tokenized_train = dataset_train.map(preprocess_function, batched=True)
tokenized_val = dataset_val.map(preprocess_function, batched=True)
tokenized_test = dataset_test.map(preprocess_function, batched=True)

In [ ]:
tokenized_train

In [ ]:
import torch
from dataclasses import dataclass
from typing import Optional, Union, Dict, List

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: any
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        labels = [f["label"] for f in features] if "label" in features[0] else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened_features = [
            {k: v[i] for k, v in f.items() if k != "label"}
            for f in features
            for i in range(num_choices)
        ]

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)

        return batch

In [ ]:
collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

In [ ]:
!pip install evaluate -q

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

In [ ]:
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer

model = AutoModelForMultipleChoice.from_pretrained("./models/ViLegalBERT")

In [ ]:
training_args = TrainingArguments(
    output_dir="mcq-lk_model",
    evaluation_strategy="steps",
    eval_steps=100,
    logging_strategy="steps",
    logging_steps=100,
    save_strategy="steps", 
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=3e-5,
    per_device_train_batch_size=12,
    per_device_eval_batch_size=12,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
test_results = trainer.predict(tokenized_test)
predictions = np.argmax(test_results.predictions, axis=1)
test_labels = test_results.label_ids

In [ ]:
predictions

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

print("Accuracy:", round(accuracy_score(test_labels, predictions)*100, 2))
print("Precision:", round(precision_score(test_labels, predictions, average='macro')*100, 2))
print("Recall:", round(recall_score(test_labels, predictions, average='macro')*100, 2))
print("F1 score:", round(f1_score(test_labels, predictions, average='macro')*100, 2))